## Salinas BESS Monthly Invoice Calculator

This notebook follows the same workflow as the prior PPOA calculator: load inputs, validate them, run the contract calculations, write outputs, and inspect the results.

In [1]:
# Imports
import sys
print(sys.executable)
import importlib
import sys
from pathlib import Path

import pandas as pd

PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Import/reload local modules in dependency order.
module_names = [
    "classes",
    "calculations",
    "data_reader",
    "data_writer",
    "error_checks",
    "report",
    "compensation_calculator",
]

for module_name in module_names:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
    else:
        importlib.import_module(module_name)

calculations = sys.modules["calculations"]
compensation_calculator = sys.modules["compensation_calculator"]
data_reader = sys.modules["data_reader"]
data_writer = sys.modules["data_writer"]
error_checks = sys.modules["error_checks"]
report = sys.modules["report"]

FORCE_MAJEURE_WAITING_PERIOD_HOURS = calculations.DEFAULT_FORCE_MAJEURE_WAITING_PERIOD_HOURS
GRID_SYSTEM_WAITING_PERIOD_HOURS = calculations.DEFAULT_GRID_SYSTEM_WAITING_PERIOD_HOURS
SCHEDULED_MAINTENANCE_ALLOWANCE_HOURS = calculations.DEFAULT_SCHEDULED_MAINTENANCE_ALLOWANCE_HOURS
calculate_monthly_results = compensation_calculator.calculate_monthly_results
load_bess_inputs = data_reader.load_bess_inputs
load_monthly_performance_guarantee_inputs = data_reader.load_monthly_performance_guarantee_inputs
load_performance_tests = data_reader.load_performance_tests
write_bess_monthly_results = data_writer.write_bess_monthly_results
input_validation = error_checks.input_validation
generate_bess_invoice_support_report = report.generate_bess_invoice_support_report

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output"

c:\Users\SamBuck\AppData\Local\Python\pythoncore-3.11-64\python.exe


### Data Inputs and Preprocessing

In [2]:
# Project and filepaths

# Change this to "jobos" to run the same workflow for Jobos.
PROJECT_ID = "salinas"

PROJECT_DATA_DIR = DATA_DIR / PROJECT_ID
OUTPUT_DIR = PROJECT_DIR / "output" / PROJECT_ID

contract_values_csv = PROJECT_DATA_DIR / "bess_contract_values_template.csv"
yearly_inputs_csv = PROJECT_DATA_DIR / "bess_yearly_inputs_template.csv"
monthly_inputs_csv = PROJECT_DATA_DIR / "bess_monthly_inputs_template.csv"
monthly_performance_guarantee_csv = PROJECT_DATA_DIR / "Monthly_Performance_Guarantee.csv"
performance_tests_csv = PROJECT_DATA_DIR / "Performance_Tests.csv"

monthly_results_csv = OUTPUT_DIR / "bess_monthly_results.csv"
report_txt = OUTPUT_DIR / "report.txt"

In [3]:
# Input file validation

input_validation(
    contract_values_csv,
    yearly_inputs_csv,
    monthly_inputs_csv,
    monthly_performance_guarantee_csv,
    performance_tests_csv,
)

print("BESS input files passed validation.")

BESS input files passed validation.


c:\Code\salinas-jobos-bess-invoice-calculator\error_checks.py:170: UserWarning: c:\Code\salinas-jobos-bess-invoice-calculator\data\salinas\bess_monthly_inputs_template.csv row 3 billing period 2026-02 has Other_ADJ=250.00. Confirm this amount does not duplicate calculator-generated ALD, CLD, or ELD.
  warnings.warn(
c:\Code\salinas-jobos-bess-invoice-calculator\error_checks.py:170: UserWarning: c:\Code\salinas-jobos-bess-invoice-calculator\data\salinas\bess_monthly_inputs_template.csv row 6 billing period 2026-05 has Other_ADJ=125.00. Confirm this amount does not duplicate calculator-generated ALD, CLD, or ELD.
  warnings.warn(
c:\Code\salinas-jobos-bess-invoice-calculator\error_checks.py:170: UserWarning: c:\Code\salinas-jobos-bess-invoice-calculator\data\salinas\bess_monthly_inputs_template.csv row 9 billing period 2026-08 has Other_ADJ=300.00. Confirm this amount does not duplicate calculator-generated ALD, CLD, or ELD.
  warnings.warn(
c:\Code\salinas-jobos-bess-invoice-calculator\

In [ ]:
# Load contract, yearly, monthly, performance guarantee, and performance test inputs

contract_values, yearly_inputs, monthly_inputs = load_bess_inputs(
    contract_values_csv,
    yearly_inputs_csv,
    monthly_inputs_csv,
)
monthly_performance_guarantee_inputs = load_monthly_performance_guarantee_inputs(
    monthly_performance_guarantee_csv
)
performance_tests = load_performance_tests(performance_tests_csv)

print(f"Loaded {len(contract_values)} contract value row(s).")
print(f"Loaded {len(yearly_inputs)} yearly input row(s).")
print(f"Loaded {len(monthly_inputs)} monthly input row(s).")
print(f"Loaded {len(monthly_performance_guarantee_inputs)} monthly performance guarantee row(s).")
print(f"Loaded {len(performance_tests)} performance test row(s).")

In [ ]:
# Preview source data

monthly_inputs_df = pd.read_csv(monthly_inputs_csv)
monthly_inputs_df.head()

### Calculations

In [ ]:
# Calculate monthly compensation results

monthly_results = calculate_monthly_results(
    contract_values,
    yearly_inputs,
    monthly_inputs,
    performance_tests,
    monthly_performance_guarantee_inputs,
)

print(f"Calculated {len(monthly_results)} monthly result row(s).")

In [ ]:
# Convert results to a review table

monthly_results_df = pd.DataFrame(
    [
        {
            "timestamp_month": result.timestamp_month,
            "agreement_year": result.agreement_year,
            "CPP": result.cpp,
            "MCC": result.mcc,
            "FA": result.fa,
            "FAA": result.faa,
            "PRA": result.pra,
            "MFP": result.mfp,
            "Other_ADJ": result.other_adj,
            "ALD": result.ald,
            "Actual_Efficiency": result.actual_efficiency,
            "ELD": result.eld,
            "ADJ_Total": result.adj_total,
            "MP": result.mp,
        }
        for result in monthly_results
    ]
)

monthly_results_df

### Annual Allowance Checks

In [ ]:
# Review annual caps and waiting periods used by FA and PRA

allowance_summary_df = (
    monthly_inputs_df
    .groupby("agreement_year", as_index=False)
    .agg(
        POHRS=("POHRS", "sum"),
        GSE=("GSE", "sum"),
        PFM=("PFM", "sum"),
    )
)

allowance_summary_df["POHRS_allowance"] = SCHEDULED_MAINTENANCE_ALLOWANCE_HOURS
allowance_summary_df["GSE_waiting_period"] = GRID_SYSTEM_WAITING_PERIOD_HOURS
allowance_summary_df["PFM_waiting_period"] = FORCE_MAJEURE_WAITING_PERIOD_HOURS
allowance_summary_df["POHRS_over_allowance"] = (
    allowance_summary_df["POHRS"] - allowance_summary_df["POHRS_allowance"]
).clip(lower=0)
allowance_summary_df["GSE_over_waiting_period"] = (
    allowance_summary_df["GSE"] - allowance_summary_df["GSE_waiting_period"]
).clip(lower=0)
allowance_summary_df["PFM_over_waiting_period"] = (
    allowance_summary_df["PFM"] - allowance_summary_df["PFM_waiting_period"]
).clip(lower=0)

allowance_summary_df

### Write Output Files

In [ ]:
# Write monthly result CSV

output_path = write_bess_monthly_results(monthly_results, output_dir=OUTPUT_DIR)
print(f"Wrote monthly results to {output_path}.")

### Output Review

In [ ]:
# Confirm the generated output file can be read back

written_results_df = pd.read_csv(monthly_results_csv)
written_results_df.head()

### Generate Invoice Support Report

This final cell calls `generate_bess_invoice_support_report()` from `report.py`.

In [ ]:
# Generate a Section 10.1-style invoice support report using report.py

report_text = generate_bess_invoice_support_report(written_results_df, report_txt)
print(f"Wrote invoice support report to {report_txt}.")
print("\n".join(report_text.splitlines()[:45]))